# Block 3b -- Addressee Tagging, Scene-Level (All Films)

Scene-level variant: one API call per scene, model annotates ALL utterances at once.
Compare with `06_addressee_tagging_all_films.ipynb` (utterance-level) to assess
whether batching affects prediction quality.

- Cache: `data/processed/{film_id}/addressee_cache_scene/{scene_id}.json`
- Output: `utterances_with_addressee_scene.csv` per film (separate from 06 output)

In [1]:
# -- CHANGE ONLY THESE LINES -----------------------------------------------
PROVIDER          = "anthropic"
MODEL             = "claude-haiku-4-5-20251001"
GROQ_API_KEY      = ""
ANTHROPIC_API_KEY = ""                        # or set in CLEAN/.env

DRY_RUN        = False
TEST_ONE_SCENE = False
FILM_FILTER    = "frozen_2013"
# ---------------------------------------------------------------------------

In [2]:
import os, json, time, re
from pathlib import Path
import pandas as pd

def _find_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if c.name == "CLEAN" and (c/"data").exists(): return c
        child = c / "CLEAN"
        if child.is_dir() and (child/"data").exists(): return child
    raise FileNotFoundError("Cannot find CLEAN root")

CLEAN_ROOT = _find_root()
CHARS_FILE = CLEAN_ROOT / "data" / "00_corpus" / "characters.csv"
PROCESSED  = CLEAN_ROOT / "data" / "02_processed"
print(f"CLEAN root : {CLEAN_ROOT}")

CLEAN root : /Users/sofapirogova/Documents/final_thesis_1106/final_master_thesis/CLEAN


In [3]:
from dotenv import load_dotenv
load_dotenv(CLEAN_ROOT / ".env")
client = None
if not DRY_RUN:
    if PROVIDER == "groq":
        from groq import Groq
        client = Groq(api_key=GROQ_API_KEY or os.environ.get("GROQ_API_KEY",""))
        print(f"Groq ready. Model: {MODEL}")
    elif PROVIDER == "anthropic":
        import anthropic as _sdk
        client = _sdk.Anthropic(api_key=ANTHROPIC_API_KEY or os.environ.get("ANTHROPIC_API_KEY",""))
        print(f"Anthropic ready. Model: {MODEL}")
else:
    print("DRY_RUN=True -- skipping client setup.")

Anthropic ready. Model: claude-haiku-4-5-20251001


In [4]:
SYSTEM_MESSAGE = (
    "You are an expert dialogue analyst annotating addressees in Disney/Pixar animated film "
    "screenplays for an academic social network analysis study. "
    "Your annotations build directed speech networks between characters. "
    "Accuracy and consistency are critical. "
    "Your output must be valid JSON only. No explanation, no markdown, no commentary."
)

VALID_TYPES  = {"individual", "group", "monologue", "non_human", "unclear"}
SPECIAL_CATS = {"group", "monologue", "non_human", "other", "unclear"}

PRIORITY_RULES = """
PRIORITY RULES (apply in order, stop at first match):
1. PARENTHETICAL: stage direction like (to Belle) -- follow it
2. DIRECT NAME/PRONOUN: speaker uses addressee's name or you
3. RESPONSE CHAIN: this utterance directly answers the previous speaker
4. SCENE PRESENCE: infer from who is physically present on-stage
5. If none apply -- unclear
"""


def build_scene_prompt(scene_id, film_id, scene_rows, char_map):
    char_lines = [f"  {cid}  ({name})" for cid, name in sorted(char_map.items())]
    char_lines += [
        "  group      -> multiple characters simultaneously",
        "  monologue  -> spoken to oneself, no listener",
        "  non_human  -> animal, object, or magical entity",
        "  unclear    -> impossible to determine",
    ]
    utt_lines = []
    for _, r in scene_rows.iterrows():
        paren = f" [{r.get('parenthetical','').strip()}]" if str(r.get("parenthetical","")).strip() else ""
        name  = char_map.get(str(r["character_id"]).strip(), str(r.get("canonical_name","")).strip())
        utt_lines.append(f"  {r['utterance_id']} | {name}{paren}: {str(r['text']).strip()[:200]}")
    ids = scene_rows["utterance_id"].tolist()
    tmpl = "[\n" + ",\n".join(
        f'  {{"utterance_id":"{uid}","addressee":"<value>","addressee_type":"<type>","confidence":"<high|medium|low>"}}'
        for uid in ids
    ) + "\n]"
    return (
        PRIORITY_RULES
        + f"\nFILM: {film_id}  |  SCENE: {scene_id}\n"
        + "\nALL UTTERANCES IN SCENE:\n" + "\n".join(utt_lines)
        + "\n\nVALID ADDRESSEE VALUES (exact string, case-sensitive):\n" + "\n".join(char_lines)
        + "\n\nFor EACH utterance above, identify the addressee using priority rules."
        + "\nReturn ONLY this JSON array (one entry per utterance, same order):\n" + tmpl
    )


def call_llm(prompt, max_retries=6):
    last_err = None
    for attempt in range(max_retries):
        try:
            if PROVIDER == "groq":
                r = client.chat.completions.create(
                    model=MODEL, temperature=0, max_tokens=4096,
                    messages=[{"role":"system","content":SYSTEM_MESSAGE},{"role":"user","content":prompt}])
                return r.choices[0].message.content, r.usage.total_tokens
            elif PROVIDER == "anthropic":
                import anthropic as _sdk
                r = client.messages.create(
                    model=MODEL, max_tokens=4096, temperature=0,
                    system=SYSTEM_MESSAGE, messages=[{"role":"user","content":prompt}])
                return r.content[0].text, r.usage.input_tokens + r.usage.output_tokens
        except Exception as e:
            last_err = e
            if any(x in str(e) for x in ["429","529","rate_limit","overloaded"]):
                wait = 15 * (2**attempt)
                print(f"  API busy -- waiting {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise last_err


def parse_scene_response(raw, expected_ids, valid_addr):
    try:
        m = re.search(r"\[.*\]", raw, re.DOTALL)
        data = json.loads(m.group(0) if m else raw)
    except Exception:
        return {uid: ("unclear","unclear","low") for uid in expected_ids}
    result = {}
    for item in data:
        uid   = str(item.get("utterance_id","")).strip()
        addr  = str(item.get("addressee","unclear")).strip()
        atype = str(item.get("addressee_type","unclear")).strip()
        conf  = str(item.get("confidence","low")).strip()
        if addr not in valid_addr:
            print(f"  warning: '{addr}' -> unclear")
            addr = "unclear"
        if atype not in VALID_TYPES:
            atype = "unclear"
        result[uid] = (addr, atype, conf)
    for uid in expected_ids:
        if uid not in result:
            result[uid] = ("unclear","unclear","low")
    return result


def cache_dir(film_dir):
    d = film_dir / "addressee_cache_scene"
    d.mkdir(exist_ok=True)
    return d

def load_cache(film_dir, scene_id):
    p = cache_dir(film_dir) / f"{scene_id}.json"
    return json.loads(p.read_text()) if p.exists() else None

def save_cache(film_dir, scene_id, prompt, raw, results):
    (cache_dir(film_dir) / f"{scene_id}.json").write_text(json.dumps(
        {"scene_id":scene_id, "prompt":prompt, "raw_response":raw,
         "results":results, "model_used":f"{PROVIDER}/{MODEL}"},
        ensure_ascii=False, indent=2))

print("Functions loaded.")

Functions loaded.


In [5]:
chars = pd.read_csv(CHARS_FILE, dtype=str).fillna("")

all_films = sorted([d.name for d in PROCESSED.iterdir()
                    if d.is_dir() and d.name != "OLD_DATA"
                    and (d/"utterances.csv").exists()])
films_to_run = [FILM_FILTER] if FILM_FILTER else all_films
print(f"Films to process: {len(films_to_run)}")

total_tokens = 0

for film_id in films_to_run:
    film_dir = PROCESSED / film_id
    out_path = film_dir / "utterances_with_addressee_scene.csv"
    df = pd.read_csv(film_dir / "utterances.csv", dtype=str).fillna("")

    film_chars = chars[chars["film_id"] == film_id]
    named = film_chars[
        film_chars["is_named"].str.lower().isin(["true","1"]) &
        ~film_chars["canonical_name"].str.contains("/", na=False)
    ]
    char_map   = dict(zip(named["character_id"].str.strip(), named["canonical_name"].str.strip()))
    valid_addr = set(char_map.keys()) | SPECIAL_CATS

    for col in ["addressee","addressee_type","confidence"]:
        if col not in df.columns: df[col] = ""

    # Auto-code songs
    song_mask = (df["addressee"].str.strip() == "") & (df["utterance_type"].str.lower() == "song")
    df.loc[song_mask, "addressee"]      = "broadcast"
    df.loc[song_mask, "addressee_type"] = "broadcast"

    needs_mask = (
        (df["addressee"].str.strip() == "") &
        (df["utterance_type"].str.lower() != "song") &
        (df["character_id"].isin(set(char_map.keys())))
    )
    scene_ids = df[needs_mask]["scene_id"].unique()
    n_cached  = sum(1 for s in scene_ids if load_cache(film_dir, s))
    n_calls   = len(scene_ids) - n_cached

    print(f"\n{'='*55}")
    print(f"Film   : {film_id}")
    print(f"Scenes : {len(scene_ids)}  (cached: {n_cached}, API calls: {n_calls})")

    if DRY_RUN:
        for scene_id in scene_ids:
            if load_cache(film_dir, scene_id): continue
            scene_rows = df[df["scene_id"]==scene_id]
            prompt = build_scene_prompt(scene_id, film_id, scene_rows, char_map)
            print(f"\n[DRY RUN] First uncached scene: {scene_id} ({len(scene_rows)} utt)")
            print("="*70 + "\nSYSTEM: " + SYSTEM_MESSAGE + "\n\nUSER PROMPT:\n" + prompt + "\n" + "="*70)
            print("\nSet DRY_RUN=False + TEST_ONE_SCENE=True to run one real call.")
            break
        continue

    for i, scene_id in enumerate(scene_ids):
        scene_rows   = df[df["scene_id"]==scene_id].copy()
        expected_ids = scene_rows["utterance_id"].tolist()

        cached = load_cache(film_dir, scene_id)
        if cached:
            for uid, vals in cached["results"].items():
                addr, atype, conf = (vals if isinstance(vals,(list,tuple)) else (vals,"unclear","low"))
                df.loc[df["utterance_id"]==uid, "addressee"]      = addr
                df.loc[df["utterance_id"]==uid, "addressee_type"] = atype
                df.loc[df["utterance_id"]==uid, "confidence"]     = conf
            continue

        if TEST_ONE_SCENE and i > 0:
            print("TEST_ONE_SCENE=True -- stopped. Set False to continue.")
            break

        prompt = build_scene_prompt(scene_id, film_id, scene_rows, char_map)
        try:
            raw, tok = call_llm(prompt)
            total_tokens += tok
            results = parse_scene_response(raw, expected_ids, valid_addr)
            save_cache(film_dir, scene_id, prompt, raw, results)
            for uid, (addr, atype, conf) in results.items():
                df.loc[df["utterance_id"]==uid, "addressee"]      = addr
                df.loc[df["utterance_id"]==uid, "addressee_type"] = atype
                df.loc[df["utterance_id"]==uid, "confidence"]     = conf
            print(f"  [{i+1:>3}/{n_calls}] {scene_id}  {len(results)} rows  [{tok:,} tok]")
            time.sleep(0.3)
        except Exception as e:
            print(f"  ERROR {scene_id}: {e}")

    df.to_csv(out_path, index=False)
    coded = (df["addressee"].str.strip() != "").sum()
    print(f"  Saved: {out_path.name}  ({coded}/{len(df)} coded)")

if not DRY_RUN:
    print(f"\nTotal tokens : {total_tokens:,}")
    print(f"Est. cost    : ~${total_tokens*0.000001*1.8:.3f} (Haiku)")

Films to process: 1

Film   : frozen_2013
Scenes : 51  (cached: 0, API calls: 51)
  [  1/51] frozen_2013_s003  6 rows  [1,268 tok]
  [  2/51] frozen_2013_s004  1 rows  [659 tok]
  [  3/51] frozen_2013_s005  16 rows  [2,349 tok]
  [  4/51] frozen_2013_s008  1 rows  [651 tok]
  [  5/51] frozen_2013_s009  1 rows  [664 tok]
  [  6/51] frozen_2013_s010  16 rows  [2,434 tok]
  [  7/51] frozen_2013_s012  20 rows  [2,976 tok]
  [  8/51] frozen_2013_s013  12 rows  [1,945 tok]
  [  9/51] frozen_2013_s014  10 rows  [1,673 tok]
  [ 10/51] frozen_2013_s015  38 rows  [5,383 tok]
  [ 11/51] frozen_2013_s017  46 rows  [5,849 tok]
  [ 12/51] frozen_2013_s018  51 rows  [6,296 tok]
  [ 13/51] frozen_2013_s019  39 rows  [5,113 tok]
  [ 14/51] frozen_2013_s020  12 rows  [1,909 tok]
  [ 15/51] frozen_2013_s021  26 rows  [3,535 tok]
  [ 16/51] frozen_2013_s022  10 rows  [2,039 tok]
  [ 17/51] frozen_2013_s023  3 rows  [974 tok]
  [ 18/51] frozen_2013_s024  3 rows  [910 tok]
  [ 19/51] frozen_2013_s025  2 row

In [6]:
# Summary
print(f"{'Film':<42} coded/total  pct")
print("-"*60)
for film_id in films_to_run:
    p = PROCESSED / film_id / "utterances_with_addressee_scene.csv"
    if not p.exists(): print(f"  {film_id}: not yet generated"); continue
    df = pd.read_csv(p, dtype=str).fillna("")
    coded = (df["addressee"].str.strip() != "").sum()
    print(f"  {film_id:<42} {coded:>4}/{len(df)}  ({100*coded/len(df):.0f}%)")

Film                                       coded/total  pct
------------------------------------------------------------
  frozen_2013                                 970/976  (99%)
